# APTOS S0-v1.0 — MOCK_TRAIN / SANITY_CHECK / FULL_TRAIN
Thiết lập một lần ở cell 1: gắn APTOS đúng `DATA_ROOT`, đặt `S0_ROOT` tới thư mục chứa `splits/`, `data_quality/`, `reproducibility/`. Danh sách tệp S0 bắt buộc nằm đầu cell 3; notebook không chứa split và không tự chia lại.

- Chọn Kaggle **GPU T4 ×2**, đổi `MODE`, rồi **Save Version → Save & Run All**. Cần Internet để tải ResNet50 V2 lần đầu, hoặc đặt `WEIGHTS_PATH` tới đúng file V2.
- `MOCK_TRAIN`: ảnh thật Train/Validation fold 0, 2 optimizer steps + 1 validation batch cho mỗi phương pháp; không Test/tuning/chọn BN.
- `SANITY_CHECK`: overfit A0/A8 trên cùng 32 ảnh Train fold 0, không augmentation, tối đa 200 steps; PASS khi accuracy=100%, loss ≤ 0,01 và giảm so với ban đầu. A8 dùng λ=0,1 (đầu grid), không tuning. A0 shuffled Train labels seed 42 chạy đủ 10 epoch; PASS theo QWK Validation epoch 10 trong [-0,1; 0,1]. Giữ `freeze_bn` trong CONFIG, không chọn BN lại. Output riêng `output/SANITY_CHECK/{training.log,sanity_report.json}`, không checkpoint/resume.
- `FULL_TRAIN`: chọn BN → tuning → khóa lựa chọn → 5-fold/Test → OOF/thống kê. Giữ 50 epoch tối đa, patience 10, batch đầy đủ 32; batch cuối giữ số ảnh còn lại.
- Resume: đặt `RESUME_ROOT` tới `output/FULL_TRAIN` của **bản refactor này**. `last.pt` giữ đầy đủ state khi run chưa xong; run hoàn tất giữ `best.pt` cho mọi trial và bỏ `last.pt` để giảm dung lượng. Không nhập checkpoint từ notebook cũ khác schema.
- Output dưới `output/<MODE>/`. S0 cần metadata acquisition/sensitivity để tính P/M và các phân tích phụ. `reports/shortcut_baseline_predictions.csv` là tùy chọn; nếu thiếu sẽ ghi `unavailable`.

Đã kiểm tra cú pháp và logic trên CPU bằng dữ liệu kiểm thử; **chưa chạy APTOS thật/T4×2**, chưa xác nhận throughput hoặc DataLoader nhiều worker trên Kaggle. Không có kết quả nghiên cứu được tạo sẵn. OOF giữ giới hạn giao thức gốc: tuning fold 0 có thể giao với Test fold khác; đây không phải nested CV. QĐ5 trong output là quyết định M1, không thay thế kiểm định Holm/DDR cho kết luận công bố.


In [27]:
# 1. CONFIG — chỉ đổi MODE sau khi gắn đúng hai dataset.
MODE = 'SANITY_CHECK'  # 'MOCK_TRAIN', 'SANITY_CHECK' hoặc 'FULL_TRAIN'
DATA_ROOT = '/kaggle/input/competitions/aptos2019-blindness-detection'
S0_ROOT = '/kaggle/input/datasets/bondyyyyy/splitaptos/splits'  # thư mục chứa splits/, data_quality/, reproducibility/
OUTPUT_ROOT = '/kaggle/working/output'
RESUME_ROOT = None  # đường dẫn chính xác tới output/FULL_TRAIN của phiên BẢN REFACTOR trước
WEIGHTS_PATH = None  # offline: đường dẫn resnet50-11ad3fa6.pth; None: cache torchvision / tải tự động
CFG = {
    'seed': 42, 'input_size': 384, 'batch_size': 32, 'num_workers': 2,
    'epochs': 50, 'patience': 10, 'lr': 1e-4, 'weight_decay': 1e-4,
    'scheduler_factor': 0.5, 'scheduler_patience': 3,
    'amp': True, 'grad_clip_norm': 5.0, 'freeze_bn': False,
    'freeze_bn_affine': False, 'a0_ce_backend': 'native',
    'checkpoint_smoothing': 3, 'crop_threshold': 8,
    'brightness': 0.1, 'contrast': 0.1, 'hflip_p': 0.5, 'vflip_p': 0.5,
    'rotation_degrees': 15.0,
    'normalization_mean': [0.485, 0.456, 0.406],
    'normalization_std': [0.229, 0.224, 0.225],
    'sqrt_beta': 0.999, 'sqrt_gamma': 2.0,
    'lambda_grid_A8': [0.1, 0.3, 0.5, 1.0, 3.0, 10.0],
    'lambda_grid_A6_A9': [0.1, 0.3, 0.5, 1.0, 3.0, 10.0],
    'alpha_grid_A12': [0.25, 0.5, 1.0, 2.0, 4.0],
    'beta_grid_A3': [0.99, 0.999, 0.9999], 'gamma_grid_A3': [1.0, 2.0],
    'slace_alpha_grid': [0.3, 0.8, 1.0, 3.0, 10.0, 25.0],
    'bootstrap_reps': 2000, 'bootstrap_seed': 20261105,
    'session_hours': 10.0, 'stop_margin_minutes': 20,
}
METHODS = ('A0', 'A3', 'A4', 'A6', 'A8', 'A9', 'A11', 'A12', 'SLACE', 'CORN')
BASELINES = ('A0', 'A3', 'A6', 'A9', 'SLACE', 'CORN')
K = 5
# Tách khỏi CFG để không đổi protocol hash / resume của MOCK và FULL.
SANITY_CFG = {'overfit_steps': 200, 'overfit_loss_threshold': 0.01, 'shuffled_epochs': 10}


In [28]:
# 2. Imports, seed, logging, atomic output.
import os, sys, json, math, time, random, hashlib, logging, shutil, gc, tempfile
from pathlib import Path
from decimal import Decimal
from concurrent.futures import ThreadPoolExecutor
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision
from torchvision import models, transforms
from scipy.optimize import minimize_scalar
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

class SessionLimit(RuntimeError): pass

def require(condition, message):
    if not condition: raise RuntimeError(message)

def clean(obj):
    if isinstance(obj, dict): return {str(k): clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, np.ndarray)): return [clean(v) for v in obj]
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, (float, np.floating)): return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, Path): return str(obj)
    return obj

def digest(obj):
    return hashlib.sha256(json.dumps(clean(obj), sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024**2), b''): h.update(chunk)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8-sig'))

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(clean(value), ensure_ascii=False, allow_nan=False), encoding='utf8')
    os.replace(temporary, path)

def atomic_torch(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.pt.tmp')
    torch.save(value, temporary); os.replace(temporary, path)

def atomic_npz(path, **arrays):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.npz.tmp')
    with open(temporary, 'wb') as f: np.savez_compressed(f, **arrays)
    os.replace(temporary, path)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

def write_metric(): atomic_json(OUT / 'metric.json', METRIC)

def check_time():
    if time.monotonic() - START_TIME >= 3600 * CFG['session_hours'] - 60 * CFG['stop_margin_minutes']:
        raise SessionLimit('Đã lưu tiến độ. Gắn output phiên này và đặt RESUME_ROOT để tiếp tục.')

def initialize_output():
    global OUT, LOG, METRIC, START_TIME, CACHE
    START_TIME = time.monotonic(); CACHE = {}
    OUT = Path(OUTPUT_ROOT) / MODE
    if MODE != 'SANITY_CHECK' and RESUME_ROOT and not (OUT / 'metric.json').exists():
        source = Path(RESUME_ROOT)
        require((source / 'metric.json').is_file(), 'RESUME_ROOT thiếu metric.json: ' + str(source))
        require(not OUT.exists() or not any(OUT.iterdir()), 'Output chưa trống; không trộn hai phiên.')
        shutil.copytree(source, OUT, dirs_exist_ok=True)
    OUT.mkdir(parents=True, exist_ok=True)
    LOG = logging.getLogger('aptos')
    for h in LOG.handlers[:]: h.close(); LOG.removeHandler(h)
    LOG.setLevel(logging.INFO); LOG.propagate = False
    for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(OUT / 'training.log', encoding='utf8')):
        h.setFormatter(logging.Formatter('%(asctime)s | %(message)s')); LOG.addHandler(h)
    METRIC = read_json(OUT / 'metric.json') if MODE != 'SANITY_CHECK' and (OUT / 'metric.json').exists() else {
        'experiment': {}, 'runs': {}, 'methods': {m: {'folds': {}} for m in METHODS},
        'oof_summary': {}, 'comparisons': {}, 'hypothesis_tests': {}, 'execution_summary': {}}
    require('runs' in METRIC and 'methods' in METRIC, 'Output thuộc notebook cũ; cần OUTPUT_ROOT mới cho bản refactor.')


In [29]:
# 3. Đọc đúng S0, kiểm split/nhãn/cluster; cache RGB đã crop-pad-resize trong RAM.
def load_s0():
    root = Path(S0_ROOT)
    required = ['splits/splits.json', 'splits/clusters.csv', 'splits/acquisition_groups.csv',
        'splits/sensitivity_test_masks_preregistered.csv', 'data_quality/image_manifest.csv',
        'data_quality/excluded_images.csv', 'reproducibility/s0_version_manifest.json',
        'reproducibility/file_hashes.json', 'reproducibility/sensitivity_preregistration.json']
    for name in required:
        require((root / name).is_file(), 'Thiếu S0-v1.0: ' + str(root / name))
    hashes = read_json(root / 'reproducibility/file_hashes.json')['sha256']
    for name in required:
        if name != 'reproducibility/file_hashes.json':
            require(name in hashes and sha_file(root / name) == hashes[name], 'S0 metadata checksum sai: ' + name)
    split = read_json(root / 'splits/splits.json')
    manifest = read_json(root / 'reproducibility/s0_version_manifest.json')
    split_hash = sha_file(root / 'splits/splits.json')
    require(split_hash == manifest['splits_json_sha256'], 'S0 split khác manifest')
    require(split_hash.startswith('d53328d2') and split_hash.endswith('394e27ca'), 'Không phải split S0-v1.0 được duyệt QĐ24')
    require(split['version'] == manifest['version'], 'S0 version mâu thuẫn')
    scenario = split['main_scenario']; folds = sorted(split['scenarios'][scenario], key=lambda f: f['fold'])
    require([f['fold'] for f in folds] == list(range(5)), 'S0 cần đúng 5 fold, chỉ số 0..4')
    def table(name):
        frame = pd.read_csv(root / name, dtype={'id_code': str}).set_index('id_code')
        require(not frame.index.duplicated().any(), 'S0 lặp ID: ' + name)
        return frame
    im = table('data_quality/image_manifest.csv'); cl = table('splits/clusters.csv')
    ac = table('splits/acquisition_groups.csv'); ex = pd.read_csv(root / 'data_quality/excluded_images.csv', dtype={'id_code': str}).set_index('id_code')
    require(set(im.index) == set(cl.index) == set(ac.index), 'Inventory S0 mâu thuẫn')
    csv = Path(DATA_ROOT) / 'train.csv'
    require(csv.is_file(), 'Thiếu file: ' + str(csv))
    source = pd.read_csv(csv, dtype={'id_code': str}).set_index('id_code')
    require('diagnosis' in source and not source.index.duplicated().any(), 'train.csv cần id_code, diagnosis duy nhất')
    require(set(source.index) == set(im.index), 'train.csv khác inventory S0')
    for labels in (source.diagnosis, cl.grade, ac.grade):
        require(np.array_equal(labels.loc[im.index], im.grade), 'Nhãn nguồn khác S0')
    require(im.grade.isin(range(K)).all(), 'Nhãn ngoài 0..4')
    eligible = set(im.index) - set(ex.index)
    require(set(cl.index[cl.excluded]) == set(ex.index), 'Exclusion S0 mâu thuẫn')
    require(len(eligible) == manifest['eligible_images'], 'Số ảnh eligible sai')
    observed_counts = im.loc[sorted(eligible)].grade.value_counts().sort_index()
    require({str(k): int(v) for k, v in observed_counts.items()} == manifest['eligible_grade_counts'], 'S0 grade counts sai')
    masks = pd.read_csv(root / 'splits/sensitivity_test_masks_preregistered.csv', dtype={'id_code': str})
    prereg = read_json(root / 'reproducibility/sensitivity_preregistration.json')
    require(prereg['no_resplitting'] and digest(masks.to_dict('records')) == prereg['masks_sha256'], 'Sensitivity masks sai')
    require(sha_file(root / 'reproducibility/sensitivity_preregistration.json') == manifest['sensitivity_preregistration_sha256'], 'Sensitivity manifest sai')
    seen = set(); partitions = []
    for fold in folds:
        parts = {p: set(fold[p]) for p in ('train', 'validation', 'test')}
        require(set.union(*parts.values()) == eligible, 'Fold không phủ đúng S0 eligible')
        for p, ids in parts.items():
            require(ids and len(ids) == len(fold[p]), 'Partition rỗng hoặc lặp ID')
            for q, other in parts.items():
                if p >= q: continue
                require(not ids & other, 'Rò rỉ ID giữa ' + p + '/' + q)
                for frame, columns in ((cl, ['exact_cluster_id', 'identity_cluster_id', 's0_cluster_id']), (im, ['pixel_sha256'])):
                    for col in columns:
                        a = frame.loc[sorted(ids), col]; b = frame.loc[sorted(other), col]
                        require(a.notna().all() and b.notna().all() and not set(a) & set(b), 'Rò rỉ/thiếu cluster: ' + col)
            partitions.append({'fold': fold['fold'], 'partition': p, 'n': len(ids)})
        require(not seen & parts['test'], 'Test folds lặp ID'); seen.update(parts['test'])
        fm = masks[masks.fold == fold['fold']]
        require(len(fm) == len(parts['test']) and set(fm.id_code) == parts['test'], 'Mask không khớp Test')
        expected = set(im.loc[sorted(parts['test'])].reset_index().drop_duplicates('pixel_sha256').id_code)
        require(set(fm.loc[fm.unique_pixel_test_keep, 'id_code']) == expected, 'Unique-pixel mask sai')
        require((fm.phash4_filtered_test_keep == ~fm.unverified_phash_le4_to_train).all(), 'pHash mask sai')
    require(seen == eligible, 'OOF Test phải phủ S0 đúng một lần')
    return {'root': root, 'folds': folds, 'images': im, 'clusters': cl, 'acquisition': ac,
        'masks': masks, 'eligible': eligible, 'scenario': scenario,
        'summary': {'version': manifest['version'], 'split_sha256': split_hash,
        'dataset_fingerprint': manifest['dataset_fingerprint'], 'eligible_count': len(eligible),
        'partitions': partitions}}

def preload(ids):
    pending = [iid for iid in ids if iid not in CACHE]
    if not pending: return
    def read_image(iid):
        path = Path(DATA_ROOT) / 'train_images' / (iid + '.png')
        require(path.is_file(), 'Thiếu ảnh: ' + str(path))
        with Image.open(path) as image: a = np.asarray(image.convert('RGB'))
        ys, xs = np.where(a.max(2) > CFG['crop_threshold'])
        require(len(xs) > 0, 'Ảnh rỗng sau crop: ' + iid)
        image = Image.fromarray(a[ys.min():ys.max()+1, xs.min():xs.max()+1])
        side = max(image.size); image = ImageOps.pad(image, (side, side), color=(0, 0, 0))
        a = np.asarray(image.resize((CFG['input_size'],) * 2, Image.Resampling.BILINEAR)).copy()
        return iid, a
    LOG.info('Cache RAM: đang đọc %d ảnh mới (chưa augmentation)', len(pending))
    with ThreadPoolExecutor(max_workers=CFG['num_workers'] or 1) as pool:
        for iid, a in pool.map(read_image, pending): CACHE[iid] = a
    LOG.info('Cache RAM: %d ảnh, %.2f GiB', len(CACHE), sum(a.nbytes for a in CACHE.values()) / 1024**3)

class FundusDataset(Dataset):
    def __init__(self, ids, s0, training=False):
        self.ids = list(ids); self.labels = s0['images'].loc[ids].grade.to_numpy(dtype=np.int64)
        self.images = [CACHE[iid] for iid in ids]; self.training = training
        self.normalize = transforms.Normalize(CFG['normalization_mean'], CFG['normalization_std'])
    def __len__(self): return len(self.ids)
    def __getitem__(self, item):
        j, epoch = item; iid = self.ids[j]; image = Image.fromarray(self.images[j])
        if self.training:
            seed = int(hashlib.sha256(f'{RUN_SEED}:{epoch}:{iid}'.encode()).hexdigest()[:16], 16)
            rnd = random.Random(seed)
            if rnd.random() < CFG['hflip_p']: image = ImageOps.mirror(image)
            if rnd.random() < CFG['vflip_p']: image = ImageOps.flip(image)
            image = image.rotate(rnd.uniform(-CFG['rotation_degrees'], CFG['rotation_degrees']), resample=Image.Resampling.BILINEAR)
            image = ImageEnhance.Brightness(image).enhance(rnd.uniform(1-CFG['brightness'], 1+CFG['brightness']))
            image = ImageEnhance.Contrast(image).enhance(rnd.uniform(1-CFG['contrast'], 1+CFG['contrast']))
        return self.normalize(transforms.functional.to_tensor(image)), int(self.labels[j]), iid


In [30]:
# 4. DataLoader tồn tại suốt run; epoch đi cùng index tới persistent workers.
# Batch đầy đủ = 32 (16/GPU); batch cuối giữ mọi mẫu còn lại, không bỏ/nhân bản ảnh.
class EpochBatches(Sampler):
    def __init__(self, n, training): self.n = n; self.training = training; self.epoch = 0
    def __len__(self): return math.ceil(self.n / CFG['batch_size'])
    def __iter__(self):
        order = np.random.default_rng(RUN_SEED + self.epoch).permutation(self.n) if self.training else np.arange(self.n)
        for start in range(0, self.n, CFG['batch_size']):
            yield [(int(j), self.epoch) for j in order[start:start + CFG['batch_size']]]

def loader(ids, s0, training=False):
    preload(ids)
    workers = min(CFG['num_workers'], os.cpu_count() or 1)
    options = {'persistent_workers': True, 'prefetch_factor': 2} if workers else {}
    return DataLoader(FundusDataset(ids, s0, training), batch_sampler=EpochBatches(len(ids), training),
        num_workers=workers, pin_memory=DEVICE.type == 'cuda',
        generator=torch.Generator().manual_seed(RUN_SEED), **options)

class GlobalAvgPool(nn.Module):
    def forward(self, x): return x.mean(dim=(2, 3), keepdim=True)

def pretrained_state():
    # Chỉ đọc weights một lần vào RAM; không dò Input hay đổi sang weights ngẫu nhiên.
    path = Path(WEIGHTS_PATH) if WEIGHTS_PATH else Path(torch.hub.get_dir()) / 'checkpoints/resnet50-11ad3fa6.pth'
    if not path.is_file():
        require(WEIGHTS_PATH is None, 'Thiếu WEIGHTS_PATH: ' + str(path))
        path.parent.mkdir(parents=True, exist_ok=True)
        try:
            torch.hub.download_url_to_file(models.ResNet50_Weights.IMAGENET1K_V2.url,
                str(path), hash_prefix='11ad3fa6', progress=False)
        except Exception as exc:
            path.unlink(missing_ok=True)
            raise RuntimeError('Cần Internet Kaggle hoặc WEIGHTS_PATH tới ResNet50 ImageNet-1K V2.') from exc
    require(sha_file(path).startswith('11ad3fa6'), 'Pretrained không đúng ResNet50 ImageNet-1K V2')
    return torch.load(path, map_location='cpu', weights_only=True)

def make_model(method, weights):
    net = models.resnet50(weights=None); net.load_state_dict(weights)
    net.avgpool = GlobalAvgPool()
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(RUN_SEED + 137)
        net.fc = nn.Linear(net.fc.in_features, K-1 if method == 'CORN' else K)
    # gamma/beta luôn trainable; frozen chỉ đóng băng running mean/variance.
    net = net.to(DEVICE)
    return nn.DataParallel(net, device_ids=[0, 1], output_device=0) if DEVICE.type == 'cuda' else net

def bare(model): return model.module if isinstance(model, nn.DataParallel) else model

def train_mode(model, frozen):
    model.train()
    if frozen:
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm): module.eval()
    # frozen=False: BN theo semantics DataParallel; running stats lưu từ replica GPU0,
    # không phải SyncBatchNorm. Cùng cách vận hành cho BN selection và mọi method.

def optimizer_bundle(model):
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max',
        factor=CFG['scheduler_factor'], patience=CFG['scheduler_patience'])
    return opt, scheduler, torch.amp.GradScaler('cuda', enabled=CFG['amp'] and DEVICE.type == 'cuda')


In [31]:
# 5. Losses: giữ nguyên công thức và tuning grid của notebook gốc.
def class_weights(counts,beta=0.999,sqrt=True,dtype=torch.float32):
    n=torch.as_tensor(counts,dtype=torch.float64)
    require(bool((n>0).all()),'Train fold thiếu class; không tự làm mượt count')
    require(0<=beta<1,'beta ngoài [0,1)')
    a=torch.ones_like(n) if beta==0 else (1-beta)/(-torch.expm1(n*math.log(beta)))
    w=a.sqrt() if sqrt else a
    return (len(n)*w/w.sum()).to(dtype)

def ordinal_parts(z,y):
    logp=F.log_softmax(z,dim=1)
    logq=torch.logcumsumexp(logp,1)[:,:-1]
    logr=torch.logcumsumexp(logp.flip(1),1).flip(1)[:,1:]
    t=y[:,None]<=torch.arange(z.shape[1]-1,device=z.device)[None,:]
    o=-torch.where(t,logq,logr).mean(1)
    # e = |CDF - t|, dùng xác suất phía sai tránh 1-exp(logq) bị cancellation.
    e=torch.where(t,logr.exp(),logq.exp())
    maximum=e.max(1,keepdim=True).values
    scaled=e/maximum.clamp_min(torch.finfo(e.dtype).tiny)
    neff=scaled.sum(1).square()/scaled.square().sum(1).clamp_min(torch.finfo(e.dtype).tiny)
    neff=torch.where(maximum[:,0]==0,torch.ones_like(neff),neff).clamp(1,z.shape[1]-1)
    d=(1+(neff-1)/(z.shape[1]-2))/2
    emd=e.square().mean(1)
    return logp,o,emd,d

def focal_vector(logp,y,w,gamma):
    lp=logp.gather(1,y[:,None]).squeeze(1)
    return -w[y]*(-torch.expm1(lp)).pow(gamma)*lp
def loss_A0(z,y,ctx):
    if CFG['a0_ce_backend']=='native':return F.cross_entropy(z,y,reduction='none'),{}
    require(CFG['a0_ce_backend']=='logsoftmax_gather','CE backend không hợp lệ')
    return -F.log_softmax(z,1).gather(1,y[:,None]).squeeze(1),{}
def loss_A3(z,y,ctx):return focal_vector(F.log_softmax(z,1),y,ctx.w,ctx.hp['gamma']),{}
def loss_A4(z,y,ctx):
    c=focal_vector(F.log_softmax(z,1),y,ctx.w,ctx.hp['gamma']);return c,{'C':c}
def loss_A6(z,y,ctx):
    lp,o,_,_=ordinal_parts(z,y);c=-lp.gather(1,y[:,None]).squeeze(1)
    return c+ctx.hp['lambda']*o,{'C':c,'O':o}
def loss_A8(z,y,ctx):
    lp,o,_,_=ordinal_parts(z,y);c=focal_vector(lp,y,ctx.w,ctx.hp['gamma'])
    return c+ctx.hp['lambda']*o,{'C':c,'O':o}
def loss_A9(z,y,ctx):
    lp,_,emd,_=ordinal_parts(z,y);c=focal_vector(lp,y,ctx.w,ctx.hp['gamma'])
    return c+ctx.hp['lambda']*emd,{'C':c,'O':emd}
def loss_A11(z,y,ctx):
    o=ordinal_parts(z,y)[1];return o,{'O':o}
def loss_A12(z,y,ctx):
    lp,o,_,d=ordinal_parts(z,y);c=focal_vector(lp,y,ctx.w,ctx.hp['gamma'])
    lam=(ctx.hp['alpha']*ctx.w[y]*d).detach()
    return c+lam*o,{'C':c,'O':o,'lambda':lam,'D':d.detach()}

def slace_tables(counts,alpha):
    n=torch.as_tensor(counts,dtype=torch.float64);k=len(n)
    require(bool((n>0).all()),'SLACE yêu cầu đủ K lớp ở train')
    prox=torch.empty(k,k,dtype=torch.float64)
    for predicted in range(k):
        for target in range(k):
            lo,hi=sorted([predicted,target]);mass=n[lo:hi+1].sum()-n[predicted]/2
            prox[predicted,target]=-torch.log(mass/n.sum())
    soft=F.softmax(alpha*prox.T,dim=1) # max(prox)-prox: hằng số triệt tiêu trong softmax
    dominance=[]
    for target in range(k):
        order=prox[:,target].argsort().flip(0)
        rank=torch.empty(k,dtype=torch.long);rank[order]=torch.arange(k)
        dominance.append(rank[None,:]<=rank[:,None])
    return soft,torch.stack(dominance)
def loss_SLACE(z,y,ctx):
    lp=F.log_softmax(z,1)
    logmass=torch.logsumexp(lp[:,None,:].masked_fill(~ctx.dom[y],-torch.inf),dim=2)
    v=-(ctx.soft[y]*logmass).sum(1)
    return v,{}
def corn_sums(z,y):
    j=torch.arange(z.shape[1],device=z.device)
    eligible=y[:,None]>=j;target=(y[:,None]>j).to(z.dtype)
    values=F.binary_cross_entropy_with_logits(z,target,reduction='none')
    return (values*eligible).sum(),eligible.sum()
def loss_CORN(z,y,ctx):
    num,den=corn_sums(z,y);return num/den,{}
def corn_decode(z):
    require(z.ndim==2 and z.shape[1]==K-1,'CORN cần conditional logits (N,K-1)')
    return (torch.sigmoid(z).cumprod(dim=1)>.5).sum(dim=1)

def decoded_predictions(p,method=None):
    p=np.asarray(p)
    # Sum the tail directly: avoids cancellation from 1-CDF near 0.5.
    return (np.cumsum(p[:,::-1],axis=1)[:,::-1][:,1:]>.5).sum(1) if method=='CORN' else p.argmax(1)

def probabilities(z,method):
    if method!='CORN':return F.softmax(z,1)
    survival=z.sigmoid().cumprod(1)
    return torch.cat([1-survival[:,:1],survival[:,:-1]-survival[:,1:],survival[:,-1:]],1)
LOSS_FUNCTIONS={m:globals()['loss_'+m] for m in METHODS}
class LossContext(nn.Module):
    def __init__(self,method,counts,hp):
        super().__init__();self.method=method;self.hp=hp
        self.register_buffer('w',class_weights(counts,hp.get('beta',CFG['sqrt_beta']),method!='A3'))
        soft,dom=slace_tables(counts,hp.get('alpha',1.0))
        self.register_buffer('soft',soft.float());self.register_buffer('dom',dom)
    def forward(self,z,y):
        # FP16 model -> FP32 loss; preserve float64 for numerical tests.
        z=z.float() if z.dtype in (torch.float16,torch.bfloat16) else z
        value,parts=LOSS_FUNCTIONS[self.method](z,y,self)
        return value.mean(),{k:v.mean() for k,v in parts.items()}
    def numerator_denominator(self,z,y):
        z=z.float() if z.dtype in (torch.float16,torch.bfloat16) else z
        if self.method=='CORN':
            n,d=corn_sums(z,y);return n,d,{}
        v,c=LOSS_FUNCTIONS[self.method](z,y,self)
        return v.sum(),len(y),{k:x.sum() for k,x in c.items()}

def parameter_grid(method):
    base={'beta':CFG['sqrt_beta'],'gamma':CFG['sqrt_gamma']}
    if method=='A3':return [{'beta':b,'gamma':g} for b in CFG['beta_grid_A3'] for g in CFG['gamma_grid_A3']]
    if method in ('A6','A8','A9'):
        grid=CFG['lambda_grid_A8'] if method=='A8' else CFG['lambda_grid_A6_A9']
        return [dict(base,**{'lambda':v}) for v in grid]
    if method=='A12':return [dict(base,alpha=a) for a in CFG['alpha_grid_A12']]
    if method=='SLACE':return [{'alpha':a,'variant':'prox_max'} for a in CFG['slace_alpha_grid']]
    return [base if method=='A4' else {}]


In [32]:
# 6. Metrics, calibration, paired bootstrap và OOF.
def metric_from_cm(cm):
    cm=np.asarray(cm,dtype=float);n=cm.sum()
    if n==0:return {'n':0,'QWK':None,'Macro_F1':None,'Accuracy':None,'MAE':None,'far_error':None}
    true=cm.sum(1);pred=cm.sum(0);tp=cm.diagonal();idx=np.arange(K)
    dist=np.abs(idx[:,None]-idx[None,:]);weights=dist**2
    expected=np.outer(true,pred)/n;den=(weights*expected).sum()
    precision=np.divide(tp,pred,out=np.zeros(K),where=pred>0)
    recall=np.divide(tp,true,out=np.zeros(K),where=true>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(K),where=precision+recall>0)
    return {'n':int(n),'QWK':float(1-(weights*cm).sum()/den) if den>0 else None,
        'Accuracy':float(tp.sum()/n),'Balanced_Accuracy':float(recall[true>0].mean()),
        'Macro_F1':float(f1.mean()),'Weighted_F1':float((f1*true).sum()/n),
        'Macro_Precision':float(precision.mean()),'Macro_Recall':float(recall.mean()),
        'MAE':float((dist*cm).sum()/n),'RMSE':float(np.sqrt((weights*cm).sum()/n)),
        'far_error':float(cm[dist>=2].sum()/n),'Recall_3':float(recall[3]) if true[3]>0 else None,
        'Recall_4':float(recall[4]) if true[4]>0 else None,
        'confusion_matrix':cm.astype(int).tolist(),
        'confusion_matrix_row_normalized':np.divide(cm,true[:,None],out=np.zeros_like(cm),where=true[:,None]>0).tolist(),
        'per_class':{str(i):{'precision':float(precision[i]),'recall':float(recall[i]),'f1':float(f1[i]),'support':int(true[i])} for i in range(K)}}
def metrics(y,p,pred=None,method=None):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=np.float64)
    require(p.shape==(len(y),K) and np.isfinite(p).all(),'Probability shape/finite sai')
    require((p>=-1e-8).all() and np.allclose(p.sum(1),1,atol=1e-5),'Probability không hợp lệ')
    pred=decoded_predictions(p,method) if pred is None else np.asarray(pred,dtype=int)
    require(pred.shape==(len(y),) and ((pred>=0)&(pred<K)).all(),'Prediction shape/range sai')
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    result=metric_from_cm(cm)
    if not len(y):return result
    conf=p[np.arange(len(y)),pred];correct=pred==y;bin_id=np.minimum((conf*15).astype(int),14)
    ece=0;bins=[]
    for b in range(15):
        ix=bin_id==b;count=int(ix.sum())
        acc=float(correct[ix].mean()) if count else None;confidence=float(conf[ix].mean()) if count else None
        if count:ece+=count/len(y)*abs(acc-confidence)
        bins.append({'bin':b,'n':count,'accuracy':acc,'confidence':confidence})
    target=y[:,None]<=np.arange(K-1)
    result.update({'NLL':float(-np.log(p[np.arange(len(y)),y].clip(1e-15)).mean()),
        'ECE15':float(ece),'RPS':float(((p.cumsum(1)[:,:-1]-target)**2).mean()),
        'Brier':float(((p-np.eye(K)[y])**2).sum(1).mean()),'calibration_bins':bins})
    return result
def scaled_probs(p,t):
    lp=np.log(np.asarray(p,dtype=float).clip(1e-15))/t;lp-=lp.max(1,keepdims=True)
    exp=np.exp(lp);return exp/exp.sum(1,keepdims=True)
def fit_temperature(y,p):
    result=minimize_scalar(lambda logt:metrics(y,scaled_probs(p,np.exp(logt)))['NLL'],bounds=(-4,4),method='bounded')
    require(result.success and np.isfinite(result.fun),'Temperature fit failed')
    t=float(np.exp(result.x))
    # T=1 thuộc miền tìm kiếm; không chấp nhận nghiệm tệ hơn gốc.
    return t if result.fun<=metrics(y,p)['NLL'] else 1.0
def confidence_interval(vals):
    vals=[x for x in vals if x is not None and np.isfinite(x)]
    return {'lower':float(np.quantile(vals,.025)) if vals else None,'upper':float(np.quantile(vals,.975)) if vals else None,'valid_replicates':len(vals)}
def paired_bootstrap(y,pa,pb,groups=None,reps=None,method_a=None,method_b=None,pred_a=None,pred_b=None):
    require(len(y)==len(pa)==len(pb),'Paired length sai')
    if not len(y):return {'status':'empty_subset'}
    reps=CFG['bootstrap_reps'] if reps is None else reps
    rng=np.random.default_rng(CFG['bootstrap_seed']);a=decoded_predictions(pa,method_a) if pred_a is None else np.asarray(pred_a);b=decoded_predictions(pb,method_b) if pred_b is None else np.asarray(pred_b)
    y=np.asarray(y);sample_groups=None
    if groups is not None:
        sample_groups=[np.flatnonzero(np.asarray(groups)==g) for g in np.unique(groups)]
    vals={k:[] for k in ['QWK','Macro_F1','MAE','far_error']}
    for _ in range(reps):
        ix=rng.integers(0,len(y),len(y)) if sample_groups is None else np.concatenate([sample_groups[g] for g in rng.integers(0,len(sample_groups),len(sample_groups))])
        ma=metric_from_cm(np.bincount(y[ix]*K+a[ix],minlength=K*K).reshape(K,K))
        mb=metric_from_cm(np.bincount(y[ix]*K+b[ix],minlength=K*K).reshape(K,K))
        for k in vals:vals[k].append(None if ma[k] is None or mb[k] is None else ma[k]-mb[k])
    ra=metrics(y,pa,pred=a);rb=metrics(y,pb,pred=b)
    return {'status':'computed','unit':'image' if groups is None else 's0_cluster','reps':reps,'seed':CFG['bootstrap_seed'],
        'n':len(y),'differences':{k:{'point':None if ra[k] is None or rb[k] is None else ra[k]-rb[k],**confidence_interval(v)} for k,v in vals.items()}}

def subgroup_masks(s0,ids):
    ac=s0['acquisition'].loc[ids];is_p=ac.size_group.isin(['1050x1050','2048x1536','819x614']).to_numpy()
    masks={'all':np.ones(len(ids),dtype=bool),'P':is_p,'M':~is_p}
    for group in sorted(ac.acquisition_group.unique()):masks['acquisition_'+str(group)]=(ac.acquisition_group==group).to_numpy()
    sm=s0['masks'].set_index('id_code').loc[ids]
    for col in ['unique_pixel_test_keep','phash4_filtered_test_keep']:masks[col]=sm[col].to_numpy(dtype=bool)
    val0=set(next(f for f in s0['folds'] if f['fold']==0)['validation'])
    masks['exclude_fold0_validation']=np.array([i not in val0 for i in ids],dtype=bool)
    if 'crosscheck_test_keep' in sm:
        masks['crosscheck_test_keep']=sm['crosscheck_test_keep'].to_numpy(dtype=bool)
    return masks
def validate_oof(s0,records):
    seen=set()
    for f,arr in records:
        ids=arr['ids'].astype(str).tolist()
        require(ids==list(f['test']),'OOF thứ tự/ID không khớp test fold')
        require(not seen&set(ids),'OOF ID lặp');seen.update(ids)
        require(np.array_equal(arr['y'],s0['images'].loc[ids].grade),'OOF nhãn sai')
        require(len(arr['p'])==len(ids),'OOF probability length sai')
    require(seen==s0['eligible'],'OOF chưa đủ dữ liệu')


In [33]:
# 7. AMP forward, FP32 loss sau DataParallel gather; không gradient accumulation.
def train_epoch(model, criterion, opt, scaler, dl, frozen):
    train_mode(model, frozen)
    totals = torch.zeros(2, device=DEVICE); components = {}; samples = 0
    for x, y, _ in dl:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=CFG['amp'] and DEVICE.type == 'cuda'):
            z = model(x)
        numerator, denominator, parts = criterion.numerator_denominator(z, y)
        require(bool(torch.isfinite(numerator)), 'Loss NaN/Inf')
        scaler.scale(numerator / denominator).backward()
        scaler.unscale_(opt)
        gradnorm = nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip_norm'], error_if_nonfinite=True)
        require(bool(torch.isfinite(gradnorm)) and bool(gradnorm > 0), 'Gradient NaN/Inf hoặc bằng 0')
        scaler.step(opt); scaler.update()
        totals[0] += numerator.detach(); totals[1] += denominator; samples += len(y)
        for name, value in parts.items(): components[name] = components.get(name, 0) + value.detach()
    num, den = totals.cpu().tolist()
    return {'loss': num / den, 'components': {k: float(v / samples) for k, v in components.items()},
        'samples': samples, 'steps': len(dl), 'effective_batch_size_nominal': CFG['batch_size'],
        'last_batch_size': samples % CFG['batch_size'] or CFG['batch_size']}

@torch.no_grad()
def infer(model, criterion, dl):
    model.eval(); ys = []; ps = []; ids = []; preds = []; totals = torch.zeros(2, device=DEVICE)
    for x, y, ii in dl:
        x = x.to(DEVICE, non_blocking=True); yg = y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=CFG['amp'] and DEVICE.type == 'cuda'):
            z = model(x)
        n, d, _ = criterion.numerator_denominator(z, yg)
        require(bool(torch.isfinite(n)), 'Validation/Test loss NaN/Inf')
        totals[0] += n; totals[1] += d
        p = probabilities(z.float(), criterion.method)
        pred = corn_decode(z.float()) if criterion.method == 'CORN' else z.argmax(1)
        ys.append(y.numpy()); ps.append(p.cpu().numpy()); preds.append(pred.cpu().numpy()); ids.extend(ii)
    arrays = {'ids': np.asarray(ids), 'y': np.concatenate(ys), 'p': np.concatenate(ps), 'pred': np.concatenate(preds)}
    num, den = totals.cpu().tolist()
    return arrays, {'loss': num / den, **metrics(arrays['y'], arrays['p'], pred=arrays['pred'])}

def train_run(run_id, method, fold, hp, frozen, s0, weights):
    signature = digest({'protocol': PROTOCOL_HASH, 'method': method, 'fold': fold['fold'], 'hp': hp, 'freeze_bn': frozen})
    folder = OUT / 'checkpoints' / run_id; folder.mkdir(parents=True, exist_ok=True)
    best_path = folder / 'best.pt'; last_path = folder / 'last.pt'
    rec = METRIC['runs'].setdefault(run_id, {'signature': signature, 'method': method,
        'fold': fold['fold'], 'hp': hp, 'freeze_bn': frozen, 'status': 'pending'})
    require(rec['signature'] == signature, 'Run khác cấu hình: ' + run_id)
    if rec['status'] == 'completed':
        validate_run(run_id, s0)
        last_path.unlink(missing_ok=True)
        LOG.info('SKIP completed %s', run_id); return rec
    check_time(); seed_all(RUN_SEED)
    model = make_model(method, weights)
    counts = np.bincount(s0['images'].loc[fold['train']].grade, minlength=K)
    ctx = LossContext(method, counts, hp).to(DEVICE)
    opt, scheduler, scaler = optimizer_bundle(model)
    train_dl = loader(fold['train'], s0, True); val_dl = loader(fold['validation'], s0)
    history = []; best = (-math.inf, -math.inf); bad = 0; start = 0; best_epoch = -1
    try:
        if last_path.exists():
            state = load_checkpoint(last_path, model, signature, opt, scheduler, scaler)
            start = state['epoch'] + 1; best = tuple(state['best_key']); bad = state['bad_epochs']
            history = state['history']; best_epoch = state['best_epoch']
            require(best_path.is_file(), 'Resume thiếu best.pt: ' + run_id)
            LOG.info('RESUME %s epoch=%d; optimizer/scheduler/scaler/RNG đã nạp', run_id, start)
        rec.update(status='training', class_weights=ctx.w.cpu().tolist(), train_class_counts=counts.tolist())
        for epoch in range(start, CFG['epochs']):
            if bad >= CFG['patience']: break
            check_time(); train_dl.batch_sampler.epoch = epoch
            t0 = time.monotonic(); lr = opt.param_groups[0]['lr']
            tr = train_epoch(model, ctx, opt, scaler, train_dl, frozen)
            train_seconds = time.monotonic() - t0; vt = time.monotonic()
            _, vm = infer(model, ctx, val_dl)
            require(vm['QWK'] is not None, 'Validation QWK undefined')
            previous = [h['validation']['QWK'] for h in history[-2:]]
            score = float(np.mean(previous + [vm['QWK']]))
            candidate = (score, -vm['NLL']); improved = candidate > best
            if improved: best = candidate; best_epoch = epoch; bad = 0
            else: bad += 1
            scheduler.step(score)
            history.append({'epoch': epoch, 'lr': lr, 'train': tr, 'validation': vm,
                'checkpoint_score': score, 'improved': improved, 'bad_epochs': bad,
                'train_seconds': train_seconds, 'validation_seconds': time.monotonic()-vt,
                'epoch_seconds': time.monotonic()-t0})
            state = checkpoint_state(model, opt, scheduler, scaler, epoch, best, bad, history, signature, best_epoch)
            if improved:
                atomic_torch(best_path, {k: state[k] for k in ('model', 'signature', 'epoch', 'best_key', 'best_epoch')})
            atomic_torch(last_path, state)
            rec.update(history=history, best_epoch=best_epoch, best_score=best[0]); write_metric()
            LOG.info('%s | epoch %d/%d lr=%.2g loss=%.5f QWK=%.5f NLL=%.5f smooth=%.5f C/O=%s sec=%.1f %s',
                run_id, epoch+1, CFG['epochs'], lr, tr['loss'], vm['QWK'], vm['NLL'], score,
                tr['components'], history[-1]['epoch_seconds'], 'BEST' if improved else '')
        load_checkpoint(best_path, model, signature)
        arrays, vm = infer(model, ctx, val_dl)
        atomic_npz(OUT / 'predictions' / (run_id + '_validation.npz'), **arrays)
        rec.update(status='completed', history=history, best_epoch=best_epoch, best_score=best[0],
            best_validation=vm, checkpoint_sha256=sha_file(best_path),
            validation_sha256=sha_file(OUT / 'predictions' / (run_id + '_validation.npz')))
        write_metric()
        # Run đã hoàn tất: best giữ cho mọi trial; last + optimizer chỉ cần khi còn dang dở.
        last_path.unlink(missing_ok=True)
        return rec
    except SessionLimit:
        rec['status'] = 'partial'; write_metric(); raise
    except Exception as exc:
        rec.update(status='failed', error=str(exc)); write_metric(); raise
    finally:
        del model, ctx, opt, scheduler, scaler, train_dl, val_dl
        gc.collect(); torch.cuda.empty_cache()


In [34]:
# 8. Checkpoint và xác minh completed. Chỉ nạp checkpoint từ output tin cậy.
def rng_state():
    return {'python': random.getstate(), 'numpy': np.random.get_state(), 'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'].cpu())
    if state['cuda']: torch.cuda.set_rng_state_all([v.cpu() for v in state['cuda']])

def checkpoint_state(model, opt, scheduler, scaler, epoch, best, bad, history, signature, best_epoch):
    return {'model': bare(model).state_dict(), 'optimizer': opt.state_dict(), 'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(), 'epoch': epoch, 'best_key': best, 'bad_epochs': bad,
        'history': history, 'signature': signature, 'best_epoch': best_epoch, 'rng': rng_state()}

def load_checkpoint(path, model, signature, opt=None, scheduler=None, scaler=None):
    state = torch.load(path, map_location='cpu', weights_only=False)
    require(state['signature'] == signature, 'Checkpoint/config/S0 không khớp: ' + str(path))
    require(state['model'] and all(bool(torch.isfinite(v).all()) for v in state['model'].values()), 'Checkpoint NaN/Inf')
    bare(model).load_state_dict(state['model'], strict=True)
    if opt is not None:
        opt.load_state_dict(state['optimizer']); scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler']); restore_rng(state['rng'])
    return state

def prediction_file(run_id, part): return OUT / 'predictions' / (run_id + '_' + part + '.npz')

def read_prediction(path, expected_ids, s0):
    require(path.is_file(), 'Thiếu prediction: ' + str(path))
    with np.load(path, allow_pickle=False) as data: arrays = {k: data[k] for k in data.files}
    require(arrays['ids'].tolist() == list(expected_ids), 'Prediction IDs sai: ' + str(path))
    require(np.array_equal(arrays['y'], s0['images'].loc[expected_ids].grade), 'Prediction labels sai')
    metrics(arrays['y'], arrays['p'], pred=arrays['pred'])
    return arrays

def validate_run(run_id, s0):
    rec = METRIC['runs'][run_id]
    require(rec['status'] == 'completed', 'Run chưa hoàn tất: ' + run_id)
    expected = digest({'protocol': PROTOCOL_HASH, 'method': rec['method'], 'fold': rec['fold'],
        'hp': rec['hp'], 'freeze_bn': rec['freeze_bn']})
    require(expected == rec['signature'], 'Completed run khác protocol: ' + run_id)
    checkpoint = OUT / 'checkpoints' / run_id / 'best.pt'; validation = prediction_file(run_id, 'validation')
    # Mỗi artifact được kiểm một lần mỗi phiên; không hash lại checkpoint ở mỗi fold.
    key = (run_id, rec['checkpoint_sha256'], rec['validation_sha256'])
    if key not in VERIFIED_RUNS:
        require(checkpoint.is_file() and sha_file(checkpoint) == rec['checkpoint_sha256'], 'Best checkpoint thiếu/đổi: ' + run_id)
        require(validation.is_file() and sha_file(validation) == rec['validation_sha256'], 'Validation artifact thiếu/đổi: ' + run_id)
        state = torch.load(checkpoint, map_location='cpu', weights_only=False)
        require(state['signature'] == expected and state['epoch'] == rec['best_epoch'], 'Best checkpoint sai signature/epoch')
        require(state['best_key'][0] == rec['best_score'], 'Best checkpoint score mâu thuẫn')
        require(all(bool(torch.isfinite(v).all()) for v in state['model'].values()), 'Checkpoint NaN/Inf')
        read_prediction(validation, s0['folds'][rec['fold']]['validation'], s0)
        VERIFIED_RUNS.add(key)
    return rec

def validate_selection_lock(lock, s0):
    require(lock.get('complete') is True and lock.get('protocol_hash') == PROTOCOL_HASH, 'Selection lock khác protocol')
    require(lock.get('split_sha256') == s0['summary']['split_sha256'], 'Selection lock khác S0')
    require(lock.get('selected_on') == 'fold0_validation_only' and lock.get('test_used') is False, 'Selection source sai')
    require(lock.get('freeze_bn') is CFG['freeze_bn'] and set(lock.get('choices', {})) == set(METHODS), 'Selection lock thiếu/sai method hoặc BN')
    for method, choice in lock['choices'].items():
        rec = validate_run(choice['run_id'], s0)
        require(rec['method'] == method and rec['fold'] == 0 and rec['freeze_bn'] is CFG['freeze_bn'], 'Selected run sai method/fold/BN')
        require(all(choice[k] == rec[k] for k in ('hp', 'signature', 'checkpoint_sha256', 'validation_sha256', 'best_score')), 'Selected trial khác lock')


In [35]:
# 9. MOCK: ảnh thật fold0, đúng 2 optimizer steps + 1 validation batch/method.
def fixed_mock_subset(s0):
    fold = s0['folds'][0]
    def take(ids, n, offset):
        rng = np.random.default_rng(RUN_SEED + offset)
        queues = [list(rng.permutation(sorted(i for i in ids if s0['images'].loc[i, 'grade'] == grade))) for grade in range(K)]
        require(len(ids) >= n, 'Không đủ mẫu thật cho MOCK')
        chosen = []
        while len(chosen) < n:
            for queue in queues:
                if queue and len(chosen) < n: chosen.append(str(queue.pop()))
        return chosen
    subset = {'train': take(fold['train'], 64, 0), 'validation': take(fold['validation'], 32, 1)}
    require(not (set(subset['train']) | set(subset['validation'])) & set(fold['test']), 'MOCK dùng nhầm Test')
    return subset

def mock_method(method, s0, weights, train_dl, val_dl):
    seed_all(RUN_SEED)
    model = make_model(method, weights)
    counts = np.bincount(s0['images'].loc[s0['folds'][0]['train']].grade, minlength=K)
    ctx = LossContext(method, counts, parameter_grid(method)[0]).to(DEVICE)
    opt, scheduler, scaler = optimizer_bundle(model)
    devices = set()
    hook = bare(model).register_forward_hook(lambda module, args, output: devices.add(output.device.index))
    try:
        first = next(model.parameters()); before = first.detach().clone()
        tr = train_epoch(model, ctx, opt, scaler, train_dl, False)
        require(tr['steps'] == 2 and tr['samples'] == 64, 'MOCK không đúng 2 steps/64 ảnh')
        require(devices == {0, 1}, 'Forward chưa thực sự chạy trên cả GPU 0 và 1')
        require(all(p.grad is not None for p in model.parameters() if p.requires_grad), 'Thiếu gradient')
        require(not torch.equal(before, first), 'Optimizer không cập nhật trọng số')
        require(len(opt.state) == sum(p.requires_grad for p in model.parameters()), 'Optimizer thiếu state')
        require(all(int(v['step']) == 2 for v in opt.state.values()), 'AMP đã bỏ qua optimizer step')
        arrays, vm = infer(model, ctx, val_dl)
        require(len(arrays['ids']) == 32 and len(val_dl) == 1, 'MOCK không đúng 1 validation batch')
        require(all(vm[k] is not None and np.isfinite(vm[k]) for k in ('loss', 'QWK', 'NLL', 'Macro_F1')), 'MOCK metric NaN/Inf')
        scheduler.step(vm['QWK'])
        signature = digest([PROTOCOL_HASH, method, 'mock'])
        with tempfile.TemporaryDirectory(dir=OUT) as tmp:
            path = Path(tmp) / 'last.pt'
            atomic_torch(path, checkpoint_state(model, opt, scheduler, scaler, 0, (vm['QWK'], -vm['NLL']), 0, [], signature, 0))
            expected_rng = (random.random(), np.random.random(), torch.rand(3))
            if DEVICE.type == 'cuda': expected_cuda = [torch.rand(3, device=f'cuda:{i}') for i in (0, 1)]
            with torch.no_grad(): first.add_(1)
            opt.param_groups[0]['lr'] = 99; scheduler.last_epoch = 99
            state = load_checkpoint(path, model, signature, opt, scheduler, scaler)
            actual_rng = (random.random(), np.random.random(), torch.rand(3))
            require(expected_rng[:2] == actual_rng[:2] and torch.equal(expected_rng[2], actual_rng[2]), 'RNG restore sai')
            if DEVICE.type == 'cuda':
                require(all(torch.equal(v, torch.rand(3, device=f'cuda:{i}')) for i, v in enumerate(expected_cuda)), 'CUDA RNG restore sai')
            require(all(torch.equal(v.cpu(), state['model'][k]) for k, v in bare(model).state_dict().items()), 'Model restore sai')
            require(opt.param_groups[0]['lr'] == state['optimizer']['param_groups'][0]['lr'] and scheduler.state_dict() == state['scheduler'], 'Optimizer/scheduler restore sai')
            require(scaler.state_dict() == state['scaler'] and state['epoch'] == 0, 'Scaler/epoch restore sai')
        return {'status': 'PASS', 'optimizer_steps': 2, 'validation_batches': 1, 'gpu_forward_devices': sorted(devices),
            'checkpoint_reload': 'PASS', 'loss': tr['loss'], 'validation': vm}
    finally:
        hook.remove(); del model, ctx, opt, scheduler, scaler
        gc.collect(); torch.cuda.empty_cache()

def mock_train(s0, weights):
    subset = fixed_mock_subset(s0)
    train_dl = loader(subset['train'], s0, True); val_dl = loader(subset['validation'], s0)
    report = {'mode': MODE, 'test_used': False, 'subset': subset, 'methods': {}}
    for method in METHODS:
        start = time.monotonic()
        try: result = mock_method(method, s0, weights, train_dl, val_dl)
        except Exception as exc:
            result = {'status': 'FAIL', 'error': str(exc)}; LOG.exception('MOCK FAIL %s', method)
        result['seconds'] = time.monotonic() - start; report['methods'][method] = result
        METRIC['mock'] = report; write_metric()
        LOG.info('MOCK %s: %s', method, result['status'])
    report['status'] = 'PASS' if all(r['status'] == 'PASS' for r in report['methods'].values()) else 'FAIL'
    write_metric()
    require(report['status'] == 'PASS', 'MOCK có FAIL; xem training.log và metric.json')


In [36]:
# SANITY_CHECK — Mục 6.4: overfit A0/A8 và shuffled labels A0; không Test/checkpoint.
def sanity_trial(name, method, shuffled, batch_ids, s0, weights, report):
    fold = s0['folds'][0]; start = time.monotonic()
    hp = parameter_grid(method)[0]  # A8: lambda=0.1 hiện tại; cố định, không tuning.
    rec = {'method': method, 'status': 'RUNNING', 'hp': hp, 'freeze_bn': CFG['freeze_bn'],
        'optimizer_steps': 0, 'epochs_completed': 0, 'loss': None, 'accuracy': None, 'QWK': None,
        'evaluation_partition': 'validation' if shuffled else 'fixed_train_batch', 'history': []}
    report['checks'][name] = rec
    model = ctx = opt = scheduler = scaler = train_dl = eval_dl = None
    try:
        seed_all(42)
        counts = np.bincount(s0['images'].loc[fold['train']].grade, minlength=K)
        model = make_model(method, weights)
        ctx = LossContext(method, counts, hp).to(DEVICE)
        opt, scheduler, scaler = optimizer_bundle(model)
        if shuffled:
            train_dl = loader(fold['train'], s0, True)
            original = train_dl.dataset.labels.copy()
            # Gán mảng mới TRƯỚC iter(loader): persistent workers nhận nhãn đã xáo.
            train_dl.dataset.labels = np.random.default_rng(42).permutation(original)
            rec['shuffle'] = {'seed': 42, 'changed_fraction': float(np.mean(original != train_dl.dataset.labels)),
                'class_counts': np.bincount(train_dl.dataset.labels, minlength=K).tolist(),
                'id_label_digest': digest(list(zip(train_dl.dataset.ids, train_dl.dataset.labels.tolist())))}
            eval_dl = loader(fold['validation'], s0)  # Nhãn Validation/S0 gốc không đổi.
            iterations = SANITY_CFG['shuffled_epochs']
            rec['criterion'] = 'QWK Validation ở epoch 10 thuộc [-0.1, 0.1]; không chọn best epoch'
        else:
            # Cache đúng một batch CPU/pinned; không augmentation hoặc đổi ảnh giữa các step.
            train_dl = [next(iter(loader(batch_ids, s0, False)))]; eval_dl = train_dl
            _, initial = infer(model, ctx, eval_dl)
            rec['initial'] = {k: initial[k] for k in ('loss', 'Accuracy', 'QWK')}
            rec['batch_ids'] = batch_ids
            iterations = SANITY_CFG['overfit_steps']
            rec['criterion'] = {'accuracy': 1.0, 'loss_max': SANITY_CFG['overfit_loss_threshold'],
                'loss_below_initial': True, 'measured_after_update': 'eval mode on the same unaugmented train batch'}
        for index in range(iterations):
            if shuffled: train_dl.batch_sampler.epoch = index
            tick = time.monotonic(); lr = opt.param_groups[0]['lr']
            tr = train_epoch(model, ctx, opt, scaler, train_dl, CFG['freeze_bn'])
            rec['optimizer_steps'] += tr['steps']
            if shuffled: rec['epochs_completed'] += 1
            _, measured = infer(model, ctx, eval_dl)
            qwk = measured['QWK']
            if shuffled:
                require(qwk is not None and np.isfinite(qwk), 'SANITY Validation QWK không hữu hạn')
                recent = [h['QWK'] for h in rec['history'][-2:]]
                scheduler.step(float(np.mean(recent + [qwk])))  # Cùng scheduler/QWK trượt như A0.
            # Overfit giữ LR gốc trong tối đa 200 steps; không scheduler theo Validation/early stopping.
            rec.update(loss=measured['loss'], accuracy=measured['Accuracy'], QWK=qwk)
            rec['history'].append({'epoch' if shuffled else 'step': index+1, 'train_loss': tr['loss'],
                'loss': rec['loss'], 'accuracy': rec['accuracy'], 'QWK': qwk, 'lr': lr,
                'optimizer_steps': rec['optimizer_steps'], 'seconds': time.monotonic()-tick})
            passed = (-0.1 <= qwk <= 0.1) if shuffled else (
                rec['accuracy'] == 1.0 and rec['loss'] <= SANITY_CFG['overfit_loss_threshold']
                and rec['loss'] < rec['initial']['loss'])
            rec['seconds'] = time.monotonic()-start
            atomic_json(OUT / 'sanity_report.json', report)
            if shuffled or (index+1) % 10 == 0 or passed:
                LOG.info('SANITY %s | %s=%d/%d loss=%.6f accuracy=%.4f QWK=%s',
                    name, 'epoch' if shuffled else 'step', index+1, iterations, rec['loss'], rec['accuracy'], qwk)
            if not shuffled and passed: break  # Chỉ dừng sớm khi đã đạt CẢ HAI tiêu chí overfit.
        rec['status'] = 'PASS' if passed else 'FAIL'
        if not passed: rec['reason'] = 'Không đạt tiêu chí sau ngân sách steps/epochs; giữ nguyên kết quả thực tế.'
    except Exception as exc:
        rec.update(status='FAIL', error_type=type(exc).__name__, reason=str(exc))
        LOG.exception('SANITY %s thất bại', name)
    finally:
        if opt is not None:
            rec['optimizer_steps'] = max((int(state.get('step', 0)) for state in opt.state.values()), default=0)
        rec['seconds'] = time.monotonic()-start
        atomic_json(OUT / 'sanity_report.json', report)
        LOG.info('SANITY %s: %s | steps=%d epochs=%d sec=%.1f',
            name, rec['status'], rec['optimizer_steps'], rec['epochs_completed'], rec['seconds'])
        del model, ctx, opt, scheduler, scaler, train_dl, eval_dl
        gc.collect(); torch.cuda.empty_cache()


def sanity_check():
    global DEVICE, RUN_SEED
    initialize_output()  # SANITY luôn chạy mới, bỏ qua RESUME_ROOT; chỉ tạo training.log.
    names = ('overfit_A0', 'overfit_A8', 'shuffled_labels_A0')
    report = {'mode': 'SANITY_CHECK', 'status': 'RUNNING', 'seed': 42, 'fold': 0,
        'test_used': False, 'config': dict(CFG), 'sanity_config': dict(SANITY_CFG),
        'near_zero_definition': 'loss <= 0.01 là ngưỡng triển khai; Review không nêu ngưỡng số.',
        'checks': {name: {'status': 'NOT_RUN', 'optimizer_steps': 0, 'epochs_completed': 0,
            'loss': None, 'accuracy': None, 'QWK': None, 'seconds': 0.0} for name in names}}
    atomic_json(OUT / 'sanity_report.json', report)
    try:
        require(torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Kaggle Settings: chọn GPU T4 ×2')
        require(all('T4' in torch.cuda.get_device_name(i) for i in (0, 1)), 'Notebook yêu cầu hai NVIDIA T4')
        DEVICE = torch.device('cuda:0'); RUN_SEED = CFG['seed']
        require(RUN_SEED == 42 and CFG['batch_size'] == 32, 'SANITY cần seed 42 và batch 32')
        require(SANITY_CFG['overfit_steps'] == 200 and SANITY_CFG['shuffled_epochs'] == 10, 'SANITY cần tối đa 200 steps / đúng 10 epoch')
        require(SANITY_CFG['overfit_loss_threshold'] > 0, 'Ngưỡng loss gần 0 phải dương')
        torch.set_num_threads(min(4, os.cpu_count() or 1)); seed_all(42)
        s0 = load_s0(); weights = pretrained_state(); fold = s0['folds'][0]
        require(len(fold['train']) >= 32, 'Train fold 0 thiếu 32 ảnh cho SANITY')
        batch_ids = np.random.default_rng(42).choice(sorted(fold['train']), 32, replace=False).tolist()
        report['split_sha256'] = s0['summary']['split_sha256']
        report['gpus'] = [torch.cuda.get_device_name(i) for i in (0, 1)]
        for name, method, shuffled in zip(names, ('A0', 'A8', 'A0'), (False, False, True)):
            sanity_trial(name, method, shuffled, batch_ids, s0, weights, report)
        report['status'] = 'PASS' if all(r['status'] == 'PASS' for r in report['checks'].values()) else 'FAIL'
    except Exception as exc:
        report.update(status='FAIL', error_type=type(exc).__name__, reason=str(exc))
        for rec in report['checks'].values():
            if rec['status'] == 'NOT_RUN': rec.update(status='FAIL', reason='Setup chưa hoàn tất: ' + str(exc))
        LOG.exception('SANITY setup thất bại')
        raise
    finally:
        report['seconds'] = time.monotonic()-START_TIME
        atomic_json(OUT / 'sanity_report.json', report)
        LOG.info('SANITY_CHECK: %s | output=%s', report['status'], OUT)
    require(report['status'] == 'PASS', 'SANITY có FAIL; xem sanity_report.json và training.log')


In [37]:
# 10. BN → tuning → khóa lựa chọn; mọi lượt ở đây chỉ đọc Train/Validation fold0.
def choose_batchnorm(scores):
    a, b = Decimal(str(scores['true'])), Decimal(str(scores['false']))
    require(np.isfinite(float(a)) and np.isfinite(float(b)), 'BN QWK không hữu hạn')
    return True if abs(a-b) < Decimal('0.01') else a > b

def select_batchnorm(s0, weights):
    file = OUT / 'bn_selection.json'; saved = read_json(file) if file.exists() else None
    if saved:
        require(saved['protocol_hash'] == PROTOCOL_HASH and saved['complete'] is True,
            'BN selection khác protocol; không tự chọn lại')
        require(saved['selected_on'] == 'fold0_validation_only' and saved['test_used'] is False, 'BN selection source sai')
    candidates = {}
    for frozen in (True, False):
        key = str(frozen).lower(); run_id = 'bn_' + key
        rec = validate_run(run_id, s0) if saved else train_run(run_id, 'A0', s0['folds'][0], {}, frozen, s0, weights)
        require(rec['method'] == 'A0' and rec['fold'] == 0 and rec['freeze_bn'] is frozen and rec['hp'] == {}, 'BN candidate sai')
        require(len(rec['history']) == CFG['epochs'] or rec['history'][-1]['bad_epochs'] >= CFG['patience'], 'BN candidate chưa hoàn tất')
        candidates[key] = {k: rec[k] for k in ('best_score', 'best_epoch', 'signature', 'checkpoint_sha256', 'validation_sha256')}
    chosen = choose_batchnorm({k: v['best_score'] for k, v in candidates.items()})
    selection = {'complete': True, 'protocol_hash': PROTOCOL_HASH, 'selected_on': 'fold0_validation_only',
        'test_used': False, 'freeze_bn': chosen, 'candidates': candidates,
        'rule': 'smoothed_QWK; absolute_delta < 0.01 => True; otherwise higher',
        'run_id': 'bn_' + str(chosen).lower()}
    require(saved is None or saved == selection, 'BN selection không khớp hai checkpoint')
    CFG['freeze_bn'] = chosen
    if saved is None: atomic_json(file, selection)
    METRIC['batchnorm_selection'] = selection; write_metric()
    LOG.info('BN LOCKED: freeze_bn=%s | True=%.6f False=%.6f', chosen, candidates['true']['best_score'], candidates['false']['best_score'])
    return selection['run_id']

def boundary_candidate(method, hp):
    # Giữ quy tắc nguồn: một điểm /3 hoặc ×3, không mở đệ quy; A3 chỉ mở gamma.
    if method == 'A3': key, grid = 'gamma', CFG['gamma_grid_A3']
    elif method in ('A6', 'A8', 'A9'):
        key, grid = 'lambda', CFG['lambda_grid_A8'] if method == 'A8' else CFG['lambda_grid_A6_A9']
    elif method == 'A12': key, grid = 'alpha', CFG['alpha_grid_A12']
    elif method == 'SLACE': key, grid = 'alpha', CFG['slace_alpha_grid']
    else: return None
    if hp[key] not in (min(grid), max(grid)): return None
    return {**hp, key: hp[key]/3 if hp[key] == min(grid) else hp[key]*3}

def tune_and_lock(s0, weights, a0_run):
    global SELECTION_LOCK
    path = OUT / 'selection_lock.json'
    if path.exists():
        SELECTION_LOCK = read_json(path); validate_selection_lock(SELECTION_LOCK, s0); return
    choices = {}; extensions = 0
    for method in METHODS:
        run_ids = []
        for index, hp in enumerate(parameter_grid(method)):
            run_id = a0_run if method == 'A0' else f'{method}_trial_{index}'
            train_run(run_id, method, s0['folds'][0], hp, CFG['freeze_bn'], s0, weights)
            run_ids.append(run_id)
        def rank(run_id):
            rec = METRIC['runs'][run_id]
            return rec['best_score'], -rec['best_validation']['NLL'], -run_ids.index(run_id)
        chosen = max(run_ids, key=rank); hp = METRIC['runs'][chosen]['hp']
        boundary = boundary_candidate(method, hp)
        if boundary is not None:
            run_id = f'{method}_trial_{len(run_ids)}'; extensions += 1
            require(extensions <= 6, 'Vượt ngân sách 6 trial biên')
            train_run(run_id, method, s0['folds'][0], boundary, CFG['freeze_bn'], s0, weights)
            run_ids.append(run_id); chosen = max(run_ids, key=rank)
        rec = METRIC['runs'][chosen]
        choices[method] = {'run_id': chosen, 'n_trials': len(run_ids), 'boundary': boundary,
            **{k: rec[k] for k in ('hp', 'signature', 'checkpoint_sha256', 'validation_sha256', 'best_score')}}
    SELECTION_LOCK = {'complete': True, 'protocol_hash': PROTOCOL_HASH, 'split_sha256': s0['summary']['split_sha256'],
        'selected_on': 'fold0_validation_only', 'test_used': False, 'freeze_bn': CFG['freeze_bn'],
        'choices': choices, 'boundary_trials': extensions}
    validate_selection_lock(SELECTION_LOCK, s0); atomic_json(path, SELECTION_LOCK)
    LOG.info('SELECTION LOCK: đã khóa đủ 10 phương pháp, chưa suy luận Test')


In [38]:
# 11. 5-fold evaluation: fold0 tham chiếu checkpoint đã chọn, không copy/train lại.
def evaluate_fold(method, fold, run_id, s0, weights):
    require(SELECTION_LOCK is not None and SELECTION_LOCK['complete'], 'Chưa khóa lựa chọn; cấm Test')
    rec = validate_run(run_id, s0); choice = SELECTION_LOCK['choices'][method]
    require(rec['hp'] == choice['hp'] and rec['freeze_bn'] is CFG['freeze_bn'], 'Evaluation khác selection lock')
    require(rec['method'] == method and rec['fold'] == fold['fold'], 'Evaluation sai method/fold')
    container = METRIC['methods'][method]['folds']; key = str(fold['fold'])
    old = container.get(key); path = prediction_file(run_id, 'test')
    if old and old.get('status') == 'completed':
        require(old['run_id'] == run_id and old['signature'] == rec['signature'], 'Evaluation resume khác run')
        require(path.is_file() and sha_file(path) == old['prediction_sha256'], 'Test prediction thiếu/đổi')
        arrays = read_prediction(path, fold['test'], s0)
        metrics(arrays['y'], arrays['p_scaled'], pred=arrays['pred_scaled'])
        LOG.info('SKIP evaluation %s fold=%d', method, fold['fold']); return
    check_time(); seed_all(RUN_SEED)
    model = make_model(method, weights)
    counts = np.bincount(s0['images'].loc[fold['train']].grade, minlength=K)
    ctx = LossContext(method, counts, rec['hp']).to(DEVICE)
    try:
        load_checkpoint(OUT / 'checkpoints' / run_id / 'best.pt', model, rec['signature'])
        val = read_prediction(prediction_file(run_id, 'validation'), fold['validation'], s0)
        temperature = fit_temperature(val['y'], val['p'])  # chỉ Validation, kể cả CORN
        arrays, raw = infer(model, ctx, loader(fold['test'], s0))
        require(arrays['ids'].tolist() == fold['test'], 'Test ID order sai')
        arrays['p_scaled'] = scaled_probs(arrays['p'], temperature)
        arrays['pred_scaled'] = decoded_predictions(arrays['p_scaled'], method)
        atomic_npz(path, **arrays)
        result = {'status': 'completed', 'run_id': run_id, 'signature': rec['signature'],
            'training_reused': fold['fold'] == 0, 'temperature': temperature, 'test': raw,
            'test_temperature_scaled': metrics(arrays['y'], arrays['p_scaled'], pred=arrays['pred_scaled']),
            'prediction_sha256': sha_file(path)}
        if method == 'CORN': result['test_argmax_auxiliary'] = metrics(arrays['y'], arrays['p'])
        container[key] = result; write_metric()
    finally:
        del model, ctx; gc.collect(); torch.cuda.empty_cache()

def full_train(s0, weights):
    a0_run = select_batchnorm(s0, weights)
    atomic_json(OUT / 'config.json', {'mode': MODE, 'config': CFG, 'protocol_hash': PROTOCOL_HASH,
        'data_root': DATA_ROOT, 's0_root': S0_ROOT, 's0': s0['summary'], 'hardware': METRIC['hardware']})
    tune_and_lock(s0, weights, a0_run)
    for method in METHODS:
        choice = SELECTION_LOCK['choices'][method]
        for fold in s0['folds']:
            run_id = choice['run_id'] if fold['fold'] == 0 else f"{method}_fold_{fold['fold']}"
            if fold['fold'] != 0:
                train_run(run_id, method, fold, choice['hp'], CFG['freeze_bn'], s0, weights)
            evaluate_fold(method, fold, run_id, s0, weights)
    summarize_experiment(s0)


In [39]:
# 12. Tổng hợp OOF, QĐ19/QĐ5, output và điểm chạy duy nhất.
def summarize_experiment(s0):
    all_predictions={};fold_rows=[]
    for m in METHODS:
        LOG.info('OOF + bootstrap %s (%d replicates)', m, CFG['bootstrap_reps'])
        records=[]
        for f in s0['folds']:
            rec=METRIC['methods'][m]['folds'].get(str(f['fold']),{})
            if rec.get('status')!='completed':break
            path=prediction_file(rec['run_id'], 'test')
            require(path.exists(),'Thiếu prediction đã hoàn thành; không bịa OOF')
            with np.load(path,allow_pickle=False) as x:arr={k:x[k] for k in x.files}
            records.append((f,arr))
            fold_rows.append({'method':m,'fold':f['fold'],**{k:v for k,v in rec['test'].items() if isinstance(v,(int,float)) or v is None}})
        if len(records)!=len(s0['folds']):continue
        validate_oof(s0,records)
        ids=np.concatenate([a['ids'] for _,a in records]);order=np.argsort(ids);ids=ids[order].astype(str)
        y=np.concatenate([a['y'] for _,a in records])[order];p=np.concatenate([a['p'] for _,a in records])[order]
        pt=np.concatenate([a['p_scaled'] for _,a in records])[order]
        pred=np.concatenate([a['pred'] for _,a in records])[order]
        pred_t=np.concatenate([a['pred_scaled'] for _,a in records])[order]
        all_predictions[m]=(ids,y,p,pt,pred,pred_t)
        atomic_npz(OUT/'predictions'/f'{m}_oof.npz',ids=ids,y=y,p=p,p_scaled=pt,pred=pred,pred_scaled=pred_t)
        masks=subgroup_masks(s0,ids.tolist())
        overall=metrics(y,p,pred=pred);fold_metrics=[METRIC['methods'][m]['folds'][str(f['fold'])]['test'] for f in s0['folds']]
        mean_sd={}
        for k,v in overall.items():
            vals=[r[k] for r in fold_metrics if isinstance(r.get(k),(float,int))]
            if len(vals)==len(fold_metrics):mean_sd[k]={'mean':float(np.mean(vals)),'sd':float(np.std(vals,ddof=1)) if len(vals)>1 else None}
        # OOF CI từng mô hình: bootstrap cùng dự đoán, tính trực tiếp QWK/MacroF1.
        rng=np.random.default_rng(CFG['bootstrap_seed']);boot={'QWK':[],'Macro_F1':[]}
        for _ in range(CFG['bootstrap_reps']):
            ix=rng.integers(len(y),size=len(y));r=metric_from_cm(np.bincount(y[ix]*K+pred[ix],minlength=K*K).reshape(K,K))
            for k in boot:boot[k].append(r[k])
        METRIC['oof_summary'][m]={'raw':overall,'temperature_scaled':metrics(y,pt,pred=pred_t),'mean_sd':mean_sd,
            'ci95':{k:confidence_interval(v) for k,v in boot.items()},
            'subgroups':{name:{'raw':metrics(y[mask],p[mask],pred=pred[mask]),'temperature_scaled':metrics(y[mask],pt[mask],pred=pred_t[mask])} for name,mask in masks.items()}}
        scaled_fold_metrics=[METRIC['methods'][m]['folds'][str(f['fold'])]['test_temperature_scaled'] for f in s0['folds']]
        METRIC['oof_summary'][m]['mean_sd_temperature_scaled']={}
        for key,value in METRIC['oof_summary'][m]['temperature_scaled'].items():
            values=[r[key] for r in scaled_fold_metrics if isinstance(r.get(key),(float,int))]
            if len(values)==len(scaled_fold_metrics):METRIC['oof_summary'][m]['mean_sd_temperature_scaled'][key]={'mean':float(np.mean(values)),'sd':float(np.std(values,ddof=1))}
        METRIC['methods'][m]['status']='completed';write_metric()
    if not all_predictions:return
    comparisons=[('A4','A3'),('A6','A0'),('A8','A4'),('A8','A9'),('A8','A11'),('A8','SLACE'),('A8','CORN'),('A12','A8')]
    for a,b in comparisons:
        LOG.info('Paired bootstrap %s - %s', a, b)
        if a not in all_predictions or b not in all_predictions:continue
        ids,y,pa,_,preda,_=all_predictions[a];bid,by,pb,_,predb,_=all_predictions[b]
        require(np.array_equal(ids,bid) and np.array_equal(y,by),'Paired OOF alignment sai')
        masks=subgroup_masks(s0,ids.tolist());res={}
        for name in ['all','M','unique_pixel_test_keep','phash4_filtered_test_keep','exclude_fold0_validation']+(['crosscheck_test_keep'] if 'crosscheck_test_keep' in masks else []):
            mask=masks[name];res[name]=paired_bootstrap(y[mask],pa[mask],pb[mask],method_a=a,method_b=b,pred_a=preda[mask],pred_b=predb[mask])
        res['cluster_all']=paired_bootstrap(y,pa,pb,s0['clusters'].loc[ids].s0_cluster_id.to_numpy(),method_a=a,method_b=b,pred_a=preda,pred_b=predb)
        METRIC['comparisons'][a+'-'+b]=res;write_metric()
    h8={'status':'blocked','reason':'requires all paired folds'}
    if 'A12-A8' in METRIC['comparisons']:
        cmp=METRIC['comparisons']['A12-A8']['all']['differences'];diffs=[];wins=0
        for f in s0['folds']:
            a=METRIC['methods']['A12']['folds'][str(f['fold'])]['test'];b=METRIC['methods']['A8']['folds'][str(f['fold'])]['test']
            diffs.append(a['Macro_F1']-b['Macro_F1']);wins+=a['QWK']>b['QWK']
        lo=cmp['QWK']['lower'];passed=len(s0['folds'])==5 and lo is not None and lo>0 and np.mean(diffs)>=0 and wins>=4
        h8={'status':'criteria_met' if passed else 'criteria_not_met','delta_qwk_ci95':cmp['QWK'],
            'mean_fold_delta_macro_f1':float(np.mean(diffs)),'winning_folds':int(wins),'required_wins':4,
            'main_method_candidate':'A12' if passed else 'A8'}
    hypothesis_map={'H1':['A4-A3'],'H2a':['A6-A0','A8-A4'],'H2b':['A8-A9'],'H3':['A8-A4','A8-A11'],'H6':['A8-SLACE']}
    for h,pairs in hypothesis_map.items():METRIC['hypothesis_tests'][h]={'status':'evidence_available_needs_interpretation' if all(p in METRIC['comparisons'] for p in pairs) else 'partial','comparisons':pairs}
    METRIC['hypothesis_tests'].update({'H4':{'status':'not_run','reason':'DDR ngoài MVP APTOS'},
        'H5':{'status':'partial','reason':'P/M metrics and paired M computed; device probe remains outside this notebook'},
        'H7':{'status':'evidence_available_needs_interpretation','reason':'raw/scaled calibration and matched-epoch validation histories available'},
        'H8':h8,'H9':{'status':'blocked' if h8['status']!='criteria_met' else 'not_run','reason':'A13 thuộc giai đoạn 2'}})
    if all(m in all_predictions for m in METHODS):
        rank_all=sorted(METHODS,key=lambda m:METRIC['oof_summary'][m]['raw']['QWK'],reverse=True)
        rank_m=sorted(METHODS,key=lambda m:METRIC['oof_summary'][m]['subgroups']['M']['raw']['QWK'] if METRIC['oof_summary'][m]['subgroups']['M']['raw']['QWK'] is not None else -float('inf'),reverse=True)
        METRIC['hypothesis_tests']['H5'].update({'rank_all':rank_all,'rank_M':rank_m,'same_ranking':rank_all==rank_m})
    shortcut=s0['root']/'reports/shortcut_baseline_predictions.csv'
    if shortcut.exists():
        expected=read_json(s0['root']/'reproducibility/file_hashes.json')['sha256'].get('reports/shortcut_baseline_predictions.csv')
        require(sha_file(shortcut)==expected,'Shortcut prediction checksum sai')
        df=pd.read_csv(shortcut,dtype={'id_code':str});df=df[(df.scenario==s0['scenario'])&(df.partition=='test')]
        baseline={}
        for m,dfm in df.groupby('model'):
            dfm=dfm.sort_values('id_code');require(set(dfm.id_code)==s0['eligible'] and not dfm.id_code.duplicated().any(),'Metadata OOF sai')
            require(np.array_equal(dfm.grade,s0['images'].loc[dfm.id_code].grade),'Metadata labels sai')
            masks=subgroup_masks(s0,dfm.id_code.tolist());y=dfm.grade.to_numpy();pred=dfm.prediction.to_numpy()
            baseline[m]={name:metric_from_cm(np.bincount(y[mask]*K+pred[mask],minlength=K*K).reshape(K,K)) for name,mask in masks.items()}
        METRIC['metadata_baselines']=baseline
    qd5_decision(s0, all_predictions)
    if not shortcut.exists(): METRIC['metadata_baselines'] = {'status': 'unavailable', 'missing_file': str(shortcut)}
    df=pd.DataFrame(fold_rows);df.to_csv(OUT/'fold_results.csv',index=False)
    scaled_rows=[{'method':m,'fold':f['fold'],**{k:v for k,v in METRIC['methods'][m]['folds'][str(f['fold'])]['test_temperature_scaled'].items() if isinstance(v,(int,float)) or v is None}} for m in METHODS for f in s0['folds'] if METRIC['methods'][m]['folds'].get(str(f['fold']),{}).get('status')=='completed']
    pd.DataFrame(scaled_rows).to_csv(OUT/'fold_results_scaled.csv',index=False)
    rows=[]
    for m,v in METRIC['oof_summary'].items():
        row={'method':m,'OOF_QWK':v['raw']['QWK']}
        for k,s in v['mean_sd'].items():row[k+'_mean']=s['mean'];row[k+'_sd']=s['sd']
        for k,stat in v['mean_sd_temperature_scaled'].items():row['scaled_'+k+'_mean']=stat['mean'];row['scaled_'+k+'_sd']=stat['sd']
        rows.append(row)
    table=pd.DataFrame(rows).sort_values('QWK_mean',ascending=False);table.to_csv(OUT/'summary.csv',index=False)
    fig,ax=plt.subplots(figsize=(10,4));ax.bar(table.method,table.QWK_mean,yerr=table.QWK_sd,capsize=3)
    ax.set(ylabel='QWK mean ± SD across folds',title='APTOS — QWK');fig.tight_layout();fig.savefig(OUT/'qwk.png',dpi=160);plt.close(fig)
    write_metric()

def qd5_decision(s0, predictions):
    # QĐ5 M1: strongest baseline xác định bằng QWK OOF, không dùng để tuning.
    require(set(predictions) == set(METHODS), 'Chưa đủ 10 phương pháp để áp dụng QĐ5')
    strongest = max(BASELINES, key=lambda m: METRIC['oof_summary'][m]['raw']['QWK'])
    main_method = METRIC['hypothesis_tests']['H8']['main_method_candidate']
    ids, y, pa, _, a, _ = predictions[main_method]
    bid, by, pb, _, b, _ = predictions[strongest]
    require(np.array_equal(ids, bid) and np.array_equal(y, by), 'QĐ5 paired alignment sai')
    comparison = paired_bootstrap(y, pa, pb, pred_a=a, pred_b=b)
    lower = comparison['differences']['QWK']['lower']
    supported = lower is not None and lower > 0
    METRIC['hypothesis_tests']['QD5'] = {
        'status': 'M1_superiority_supported' if supported else 'benchmark_direction',
        'main_method': main_method, 'baseline_candidates': BASELINES, 'strongest_baseline': strongest,
        'baseline_selected_by': 'highest_raw_OOF_QWK', 'paired_comparison': comparison,
        'rule': 'QĐ5 M1: positive delta QWK with 95% paired-bootstrap CI entirely above zero'}

def main():
    global DEVICE, RUN_SEED, PROTOCOL_HASH, SELECTION_LOCK, VERIFIED_RUNS
    require(MODE in ('MOCK_TRAIN', 'SANITY_CHECK', 'FULL_TRAIN'), 'MODE phải là MOCK_TRAIN, SANITY_CHECK hoặc FULL_TRAIN')
    if MODE == 'SANITY_CHECK':
        sanity_check()
        return
    initialize_output(); SELECTION_LOCK = None; VERIFIED_RUNS = set()
    try:
        require(torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Kaggle Settings: chọn GPU T4 ×2')
        require(all('T4' in torch.cuda.get_device_name(i) for i in (0, 1)), 'Notebook yêu cầu hai NVIDIA T4')
        DEVICE = torch.device('cuda:0'); RUN_SEED = CFG['seed']
        require(RUN_SEED == 42 and CFG['batch_size'] == 32 and not CFG['freeze_bn_affine'], 'QĐ25: seed 42, batch 32, BN affine trainable')
        require(CFG['checkpoint_smoothing'] == 3, 'Checkpoint smoothing phải là 3 epoch')
        if MODE == 'FULL_TRAIN':
            require(CFG['epochs'] == 50 and CFG['patience'] == 10, 'FULL giữ tối đa 50 epoch, patience 10')
        torch.set_num_threads(min(4, os.cpu_count() or 1)); seed_all(RUN_SEED)
        s0 = load_s0()
        config = {k: v for k, v in CFG.items() if k not in ('freeze_bn', 'num_workers', 'session_hours', 'stop_margin_minutes')}
        PROTOCOL_HASH = digest({'schema': 'aptos-refactor-1', 'mode': MODE, 'config': config,
            'split': s0['summary']['split_sha256'], 'pretrained': 'resnet50-11ad3fa6',
            'batchnorm': 'DataParallel-local; QD25-smoothed-QWK-0.01', 'boundary': 'one-point-x3; A3-gamma'})
        previous = METRIC['experiment'].get('protocol_hash')
        require(previous in (None, PROTOCOL_HASH), 'Output khác MODE/config/S0 hoặc bản notebook. Dùng OUTPUT_ROOT mới.')
        METRIC['experiment'].update(protocol_hash=PROTOCOL_HASH, mode=MODE, status='running', data=s0['summary'])
        METRIC['hardware'] = {'gpus': [torch.cuda.get_device_name(i) for i in (0, 1)],
            'torch': torch.__version__, 'torchvision': torchvision.__version__, 'cuda': torch.version.cuda,
            'parallel': 'DataParallel(device_ids=[0,1])', 'num_workers': CFG['num_workers']}
        atomic_json(OUT / 'config.json', {'mode': MODE, 'config': CFG, 'protocol_hash': PROTOCOL_HASH,
            'data_root': DATA_ROOT, 's0_root': S0_ROOT, 's0': s0['summary'], 'hardware': METRIC['hardware']})
        write_metric(); LOG.info('%s | seed=%d | GPUs=%s', MODE, RUN_SEED, METRIC['hardware']['gpus'])
        weights = pretrained_state()
        if MODE == 'MOCK_TRAIN': mock_train(s0, weights)
        else: full_train(s0, weights)
        METRIC['experiment']['status'] = 'completed'
    except SessionLimit as exc:
        METRIC['experiment']['status'] = 'partial'; LOG.warning('%s', exc)
    except Exception as exc:
        METRIC['experiment'].update(status='failed', error=str(exc)); LOG.exception('Dừng: %s', exc)
        raise
    finally:
        runs = list(METRIC['runs'].values()); rows = [h for r in runs for h in r.get('history', [])]
        seconds = sum(h['train_seconds'] for h in rows)
        METRIC['execution_summary'].update(
            completed_training_runs=sum(r['status'] == 'completed' for r in runs),
            reused_fold0_records=sum(r.get('training_reused', False) for m in METRIC['methods'].values() for r in m['folds'].values()),
            training_seconds=seconds, validation_seconds=sum(h['validation_seconds'] for h in rows),
            training_images_per_second=sum(h['train']['samples'] for h in rows)/seconds if seconds else None,
            total_runtime_seconds=METRIC['execution_summary'].get('total_runtime_seconds', 0) + time.monotonic()-START_TIME,
            gpu_peak_allocated_gb=[torch.cuda.max_memory_allocated(i)/1024**3 for i in range(torch.cuda.device_count())])
        write_metric(); LOG.info('Trạng thái: %s | output=%s', METRIC['experiment']['status'], OUT)

if __name__ == '__main__':
    main()


2026-11-14 08:57:39,328 | Cache RAM: đang đọc 32 ảnh mới (chưa augmentation)
2026-11-14 08:57:47,289 | Cache RAM: 32 ảnh, 0.01 GiB
2026-11-14 08:57:59,298 | SANITY overfit_A0 | step=10/200 loss=1.250872 accuracy=0.6250 QWK=0.33333333333333337
2026-11-14 08:58:03,069 | SANITY overfit_A0 | step=20/200 loss=0.533201 accuracy=0.8750 QWK=0.7277486910994764
2026-11-14 08:58:06,943 | SANITY overfit_A0 | step=30/200 loss=0.118546 accuracy=0.9688 QWK=0.9124087591240876
2026-11-14 08:58:10,958 | SANITY overfit_A0 | step=40/200 loss=0.021178 accuracy=1.0000 QWK=1.0
2026-11-14 08:58:13,604 | SANITY overfit_A0 | step=47/200 loss=0.009404 accuracy=1.0000 QWK=1.0
2026-11-14 08:58:13,607 | SANITY overfit_A0: PASS | steps=47 epochs=0 sec=35.2
2026-11-14 08:58:19,225 | SANITY overfit_A8 | step=10/200 loss=0.668553 accuracy=0.6562 QWK=0.39158576051779936
2026-11-14 08:58:23,085 | SANITY overfit_A8 | step=20/200 loss=0.259201 accuracy=0.9375 QWK=0.87409200968523
2026-11-14 08:58:26,926 | SANITY overfit_A8